## a) Basic setup
1. python packages
2. API keys

In [ ]:
!pip install -q biopython requests pandas dotenv

In [ ]:
import os

# Detect environment and load secrets
try:
    from google.colab import userdata

    # --- Google Colab ---
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    ENTREZ_EMAIL = userdata.get("ENTREZ_EMAIL")
    NCBI_API_KEY = userdata.get("NCBI_API_KEY")

    print("Running in Google Colab.")

except ImportError:
    # --- Local Jupyter ---
    from dotenv import load_dotenv

    load_dotenv()  # reads .env from the notebook's directory

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
    ENTREZ_EMAIL = os.getenv("ENTREZ_EMAIL")
    NCBI_API_KEY = os.getenv("NCBI_API_KEY")

    print("Running locally.")

# Validate required secrets
assert GEMINI_API_KEY, (
    "GEMINI_API_KEY is not configured -- get a free key at https://aistudio.google.com/api-keys"
)

assert ENTREZ_EMAIL, (
    "ENTREZ_EMAIL is not configured."
)

if NCBI_API_KEY:
    print("NCBI_API_KEY found.")
else:
    print("No NCBI_API_KEY found -- using the slower NCBI rate limit.")


In [ ]:
from Bio import Entrez

Entrez.email = ENTREZ_EMAIL
if NCBI_API_KEY:
    Entrez.api_key = NCBI_API_KEY

# NCBI rate limit: ~3 req/s without an API key, ~10 req/s with one.
NCBI_SLEEP = 0.11 if NCBI_API_KEY else 0.4


---
## Before you start

Picks up after [02_triage.ipynb](02_triage.ipynb). Triage filters which papers are worth
curating; this notebook extracts structured data (strains, genes, resistance phenotypes)
from the same `sample_pmids`. Fetches its own papers, so it also runs standalone.

## b) The Gemini call

`call_llm` sends one prompt+paper-text pair to the Gemini API and parses the `{"answer": ..., "reasoning": ...}` JSON out of the response. `run_triage` walks `TRIAGE_FLOWCHART` node by node: at each `conditional_prompt_boolean` node call the LLM and branch on `true`/`false`, stop at `terminal_triage`.

In [ ]:
# SET the MODEL NAME you would like to use via the Gemini API, and its free-tier
# RPM (requests/minute) and RPD (requests/day) caps -- check current limits at
# https://ai.google.dev/gemini-api/docs/rate-limits if this model's caps change.
GEMINI_URL_TEMPLATE = "https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
GEMINI_MODEL = "gemini-3.5-flash-lite"
GEMINI_RPM = 15
GEMINI_RPD = 500

MIN_REQUEST_INTERVAL = 60.0 / GEMINI_RPM  # spaced out evenly across a minute
_last_call_end = 0.0
_daily_request_count = 0


In [ ]:
import json

# Edit prompt wording directly here -- this is the one and only place these questions are
# defined (no download/checksum step). Re-run this cell after any edit for it to take effect.
# target_section lists which paper section(s) (IAO ontology codes) this question should see when
# triaging full text, section-aware (section e) -- TITLE=IAO:0000305, ABSTRACT=IAO:0000315,
# METHODS=IAO:0000317, RESULTS=IAO:0000318, BODY=IAO:0000314 (a catch-all for untyped body text).
TRIAGE_FLOWCHART = {
    "startNode": "check_bacteria",
    "nodes": {
        "check_bacteria": {
            "type": "conditional_prompt_boolean",
            "data": {"prompt_name": "is_about_bacteria"},
            "transitions": {"true": "check_gene_genotype", "false": "terminal_triage"},
        },
        "check_gene_genotype": {
            "type": "conditional_prompt_boolean",
            "data": {"prompt_name": "has_gene_genotype"},
            "transitions": {"true": "check_amr_phenotype", "false": "check_amr_phenotype"},
        },
        "check_amr_phenotype": {
            "type": "conditional_prompt_boolean",
            "data": {"prompt_name": "has_amr_phenotype"},
            "transitions": {"true": "terminal_triage", "false": "terminal_triage"},
        },
        "terminal_triage": {"type": "terminal_triage", "data": {}, "transitions": {}},
    },
}

_PROMPTS_RAW = {
    "prompts": [
        {
            "name": "is_about_bacteria",
            "target_section": ["IAO:0000305", "IAO:0000315", "IAO:0000317", "IAO:0000314"],
            "prompt": "Does this paper study bacteria or bacterial strains?",
            "output_format": "boolean",
        },
        {
            "name": "has_gene_genotype",
            "target_section": ["IAO:0000305", "IAO:0000315", "IAO:0000317", "IAO:0000318", "IAO:0000314"],
            "prompt": ("Does this paper report a specific bacterial gene or genotype by name (e.g. a named "
                       "resistance gene, efflux pump gene, or genotype/lineage designation) in the bacterial "
                       "strain(s) under study, even if no specific sequence variant/mutation is described? "
                       "Answer NO if no gene or genotype is named for the study bacterium."),
            "output_format": "boolean",
        },
        {
            "name": "has_amr_phenotype",
            "target_section": ["IAO:0000305", "IAO:0000315", "IAO:0000317", "IAO:0000318", "IAO:0000314"],
            "prompt": ("Does this paper report a phenotype specifically related to antimicrobial/antibiotic "
                       "resistance or susceptibility for the bacterial strain(s) under study (e.g. MIC values, "
                       "resistant/susceptible/intermediate classification, resistance gene expression, inhibition "
                       "zone diameter)? Answer NO if the only phenotypes reported are unrelated to antimicrobial "
                       "resistance/susceptibility (e.g. growth rate, biofilm formation, virulence with no "
                       "antimicrobial/antibiotic resistance angle)."),
            "output_format": "boolean",
        },
    ],
}

TRIAGE_PROMPTS = {p["name"]: p for p in _PROMPTS_RAW["prompts"]}

print("Start node:", TRIAGE_FLOWCHART["startNode"])
print("Questions:", list(TRIAGE_PROMPTS.keys()))


In [ ]:
import re
import time

# Gemini can return 429 (per-minute/per-day quota) or a transient 5xx -- both are worth
# retrying with backoff rather than failing the whole run on one blip.
RETRYABLE_STATUS_CODES = {429, 500, 502, 503}


def _extract_json(text):
    """Model output is usually pure JSON, but strip stray markdown fences/prose defensively."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(match.group(0)) if match else {"answer": "no", "reasoning": f"unparsed: {text[:200]}"}


def _gemini_generate(user_content, model=None, max_tokens=8192, max_retries=5):
    """Send one prompt to the Gemini API and return the raw text response -- the shared
    HTTP/retry/rate-limit plumbing behind call_llm_curation (structured JSON extraction,
    section f). Callers own parsing. `model` defaults to GEMINI_MODEL (above) if not given --
    pass a different model id (e.g. CURATION_GEMINI_MODEL, section f) to use a different one.
    `max_tokens` caps the response (Gemini's own default cap is too small for a multi-field
    strains/genes JSON array -- that's what was silently truncating curation output)."""
    global _last_call_end, _daily_request_count

    if _daily_request_count >= GEMINI_RPD:
        raise RuntimeError(
            f"Reached the configured daily cap (GEMINI_RPD={GEMINI_RPD}) for this session -- "
            "Gemini would reject further requests anyway. Wait until tomorrow, raise GEMINI_RPD "
            "if your actual quota is higher, or switch to the local model (section e)."
        )

    wait = MIN_REQUEST_INTERVAL - (time.time() - _last_call_end)
    if wait > 0:
        time.sleep(wait)

    url = GEMINI_URL_TEMPLATE.format(model=model or GEMINI_MODEL)

    for attempt in range(max_retries):
        try:
            resp = requests.post(
                url,
                params={"key": GEMINI_API_KEY},
                json={
                    "contents": [{"parts": [{"text": user_content}]}],
                    "generationConfig": {"temperature": 0, "maxOutputTokens": max_tokens},
                },
                timeout=120,
            )
        except requests.exceptions.RequestException as e:
            # A read timeout / connection drop never reaches resp.status_code -- without this,
            # one network blip raises straight out of here and (depending on the caller) can
            # abort an entire multi-paper loop instead of just this one call. Retry it exactly
            # like a transient 5xx.
            _last_call_end = time.time()
            backoff = 5.0 * (2 ** attempt)
            print(f"  network error ({e!r}), backing off {backoff:.0f}s "
                  f"(attempt {attempt + 1}/{max_retries})...")
            time.sleep(backoff)
            continue
        _last_call_end = time.time()
        _daily_request_count += 1

        if resp.status_code in RETRYABLE_STATUS_CODES:
            backoff = 5.0 * (2 ** attempt)
            print(f"  {resp.status_code} ({resp.text[:200]!r}), backing off {backoff:.0f}s "
                  f"(attempt {attempt + 1}/{max_retries})...")
            time.sleep(backoff)
            continue

        if not resp.ok:
            # Surface the actual reason (bad model id, invalid key, etc.) instead of a bare
            # HTTPError -- Gemini's error body usually says exactly what's wrong.
            print(f"  Gemini error {resp.status_code}: {resp.text[:500]}")
            resp.raise_for_status()

        return resp.json()["candidates"][0]["content"]["parts"][0]["text"]

    raise RuntimeError(
        "Gemini call failed after retries -- likely your free-tier rate limit (requests/minute "
        "or requests/day). Wait a bit and retry."
    )


def call_llm(question_prompt, paper_text, max_retries=5):
    user_content = (
        f"{question_prompt}\n\n"
        f"Paper text:\n{paper_text[:6000]}\n\n"
        'Respond ONLY with JSON in exactly this shape: {"answer": "yes" or "no", "reasoning": "one sentence"}'
    )
    content = _gemini_generate(user_content, max_retries=max_retries)
    parsed = _extract_json(content)
    answer = str(parsed.get("answer", "")).strip().lower() == "yes"
    return answer, parsed.get("reasoning", "")


def run_triage(paper_text, verbose=True):
    node_name = TRIAGE_FLOWCHART["startNode"]
    extraction_data = {}

    while True:
        node = TRIAGE_FLOWCHART["nodes"][node_name]
        if node["type"] == "terminal_triage":
            break

        prompt_name = node["data"]["prompt_name"]
        question_prompt = TRIAGE_PROMPTS[prompt_name]["prompt"]

        answer, reasoning = call_llm(question_prompt, paper_text)
        extraction_data[prompt_name] = answer
        if verbose:
            print(f"  [{prompt_name}] {answer} — {reasoning}")

        node_name = node["transitions"]["true" if answer else "false"]

    return {
        "is_bacteria": extraction_data.get("is_about_bacteria"),
        "has_gene_genotype": extraction_data.get("has_gene_genotype"),
        "has_amr_phenotype": extraction_data.get("has_amr_phenotype"),
    }


### Offline fallback: bundled sample papers

`sample_fulltext/` has pre-fetched text for the sample PMIDs, plus a `manifest.json` of
PMID -> PMCID. Used only if NCBI is unreachable — the fetch functions below always try
Entrez first. Doesn't cover PMIDs you add yourself.

In [ ]:
import json as _json

FALLBACK_DIR = "sample_fulltext"
with open(f"{FALLBACK_DIR}/manifest.json") as f:
    FALLBACK_MANIFEST = _json.load(f)

print(f"Loaded offline fallback manifest for {len(FALLBACK_MANIFEST)} PMIDs.")


## c) Get sample papers

The pipeline triages full-text BioC sections; for this self-contained demo we fetch title+abstract from PubMed for a handful of known AMR papers via `Bio.Entrez` — a reasonable simplification for a workshop exercise, not what the production pipeline does (it sees full text, and uses PMC's BioC XML/JSON export, not just the abstract).

In [ ]:
# A small curated set of real PMIDs — a mix of clear AMR papers and one that isn't AMR-focused,
# so triage has something to actually discriminate on.
#sample_pmids = ["12737743", "12800222", "12927083", "12059906"]
sample_pmids = ['12737743', '12800222', '12927083', '12059906','34515028', '34485958', '34463568', '34461978', '34414415'] # these are reported in AMR portal


def fetch_title_abstract(pmid):
    """Falls back to sample_fulltext/ (see above) if NCBI is unreachable."""
    try:
        handle = Entrez.efetch(db="pubmed", id=pmid, rettype="abstract", retmode="xml")
        records = Entrez.read(handle)
        handle.close()
        article = records["PubmedArticle"][0]["MedlineCitation"]["Article"]
        title = str(article.get("ArticleTitle", ""))
        abstract_parts = article.get("Abstract", {}).get("AbstractText", [])
        abstract = "\n".join(str(p) for p in abstract_parts)
        return f"{title}\n\n{abstract}"
    except Exception as e:
        local_path = f"{FALLBACK_DIR}/{pmid}_title_abstract.txt"
        if os.path.exists(local_path):
            print(f"  NCBI fetch failed ({e!r}) -- using bundled local copy for PMID {pmid}.")
            with open(local_path) as f:
                return f.read()
        raise


papers = {}
for pmid in sample_pmids:
    papers[pmid] = fetch_title_abstract(pmid)
    time.sleep(NCBI_SLEEP)

print(f"Fetched {len(papers)} papers")
print(papers[sample_pmids[0]][:400])

---
## d) Fetch full-text sections

`run_curation_sectioned` (section f) curates each stage from the paper section(s) listed in its
own `target_section` (IAO ontology codes) instead of one flat title+abstract blob, the same
section-aware approach 02_triage.ipynb uses. This fetches a paper's full text from PMC as JATS
XML and splits it into `{iao_id: text}` by heading. Only works for PMC's open-access full-text
subset -- a PMID with no PMC link, or a metadata-only record, comes back with few or no
sections, and curation falls back to title+abstract for it.

In [ ]:
# PMC's Open Access subset, pre-segmented into passages by NCBI's own text-mining pipeline
# (https://www.ncbi.nlm.nih.gov/research/bionlp/APIs/BioC-PMC/) -- infons["section_type"] is a
# clean enum (TITLE/ABSTRACT/INTRO/METHODS/RESULTS/DISCUSS/FIG/TABLE/...), so no hand-rolled
# heading-substring matching is needed the way JATS XML parsing used to require. Anything
# outside the core narrative sections (FIG/TABLE/REF/...) is bucketed into the generic
# IAO:0000314 "body" code -- an inline TABLE passage can still carry real MIC/susceptibility
# data worth exposing to the genes/phenotypes curation stages.
SECTION_TYPE_TO_IAO = {
    "TITLE": "IAO:0000305",
    "ABSTRACT": "IAO:0000315",
    "INTRO": "IAO:0000316",
    "METHODS": "IAO:0000317",
    "RESULTS": "IAO:0000318",
    "DISCUSS": "IAO:0000319",
}
BODY_IAO = "IAO:0000314"

BIOC_CACHE_DIR = "bioc_cache"


def pmid_to_pmcid(pmid):
    """Looks up the PMC ID linked to a PubMed PMID; None if there isn't one.
    Falls back to the local manifest if NCBI is unreachable."""
    try:
        handle = Entrez.elink(dbfrom="pubmed", db="pmc", id=pmid)
        result = Entrez.read(handle)
        handle.close()
        return f"PMC{result[0]['LinkSetDb'][0]['Link'][0]['Id']}"
    except (IndexError, KeyError):
        return None
    except Exception as e:
        entry = FALLBACK_MANIFEST.get(pmid)
        if entry:
            print(f"  NCBI elink failed ({e!r}) -- using bundled manifest for PMID {pmid}.")
            return entry.get("pmcid")
        return None


def _bioc_fulltext_json(pmcid):
    """Fetch a PMC article as BioC JSON and cache it to bioc_cache/ so later cells -- and
    other notebooks run from this same working directory -- don't refetch it. Falls back to
    the bundled sample_fulltext/ copy if NCBI is unreachable or the article isn't in PMC's
    Open Access BioC subset (narrower than what Entrez.efetch(db="pmc") alone would serve --
    some PMCIDs come back with title+abstract only, same limitation the old JATS-XML fetch
    had for these same papers)."""
    os.makedirs(BIOC_CACHE_DIR, exist_ok=True)
    cache_path = os.path.join(BIOC_CACHE_DIR, f"{pmcid}_fulltext.json")
    if os.path.exists(cache_path):
        return json.load(open(cache_path))

    url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode"
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        data = json.loads(resp.text)
        # NCBI wraps the BioCCollection in a top-level list, even for a single document.
        if not (isinstance(data, list) and data and data[0].get("documents")):
            raise ValueError("empty/unexpected BioC response")
        json.dump(data, open(cache_path, "w"))
        return data
    except Exception as e:
        local_path = f"{FALLBACK_DIR}/{pmcid}_fulltext.json"
        if os.path.exists(local_path):
            print(f"  NCBI BioC fetch failed ({e!r}) -- using bundled local copy for {pmcid}.")
            return json.load(open(local_path))
        return None


def fetch_sections_from_pmc(pmcid):
    """Fetches a PMC article as BioC JSON (see _bioc_fulltext_json) and groups passage text
    by IAO section id via SECTION_TYPE_TO_IAO. Returns {} if the article isn't in PMC's Open
    Access BioC subset and no local fallback exists."""
    data = _bioc_fulltext_json(pmcid)
    if not data:
        return {}

    sections = {}
    for collection in data:
        for doc in collection.get("documents", []):
            for passage in doc.get("passages", []):
                text = passage.get("text", "")
                if not text:
                    continue
                section_type = passage.get("infons", {}).get("section_type", "")
                iao_id = SECTION_TYPE_TO_IAO.get(section_type, BODY_IAO)
                sections[iao_id] = (sections.get(iao_id, "") + "\n" + text).strip()

    return sections


def select_target_text(sections, target_section_ids, max_chars=30000):
    """Concatenate only the sections a prompt cares about; fall back to
    everything available if none of its target sections were found."""
    matched = [sections[i] for i in target_section_ids if i in sections]
    text = "\n\n".join(matched) if matched else "\n\n".join(sections.values())
    return text[:max_chars]


def run_triage_sectioned(sections, verbose=True):
    """Same flowchart walk as run_triage, but each question only sees the
    section(s) listed in its own target_section instead of one flat blob."""
    node_name = TRIAGE_FLOWCHART["startNode"]
    extraction_data = {}

    while True:
        node = TRIAGE_FLOWCHART["nodes"][node_name]
        if node["type"] == "terminal_triage":
            break

        prompt_name = node["data"]["prompt_name"]
        prompt_def = TRIAGE_PROMPTS[prompt_name]
        paper_text = select_target_text(sections, prompt_def.get("target_section", []))

        answer, reasoning = call_llm(prompt_def["prompt"], paper_text)
        extraction_data[prompt_name] = answer
        if verbose:
            print(f"  [{prompt_name}] {answer} — {reasoning}")

        node_name = node["transitions"]["true" if answer else "false"]

    return {
        "is_bacteria": extraction_data.get("is_about_bacteria"),
        "has_gene_genotype": extraction_data.get("has_gene_genotype"),
        "has_amr_phenotype": extraction_data.get("has_amr_phenotype"),
    }

---
## e) Curation: structured extraction beyond triage

Triage (02_triage.ipynb) only answers yes/no questions. Curation goes further: for a paper that
passed triage, extract the actual AMR data -- which bacterial strains, which genes, and which
experimentally-measured AMR phenotypes -- as structured records, following a **species -> gene ->
phenotype** chain. A separate per-mutation variant stage was deliberately left out: triage already
dropped its own `has_phenotype_data` node once its `true`/`false` transitions both pointed at the
same next node (see `CLAUDE.md`'s "Known-trimmed node"), and the same reasoning applies here --
for the AMR-specific lens this notebook cares about (resistance gene, resistance mechanism,
antibiotic, MIC/susceptibility call), the gene **is** the unit a phenotype attaches to; a separate
variant stage would mostly be bookkeeping (`variant_id`) with no field downstream that only a
variant, and not its gene, can carry. Each stage's prompt sees the paper text *and* the previous
stage's output, so the genes stage can only name organisms the strains stage already found, and
phenotypes can only reference genes the genes stage already found.

Like `TRIAGE_PROMPTS`, `CURATION_PROMPTS` below is a plain Python dict defined directly in this
notebook -- edit a stage's wording here and re-run, no download/checksum round-trip. The field
vocabulary (`gene_class`, `resistance_mechanism`, `confers_resistance`, `mic_value`,
`susceptibility_call`, ...) and the "do not infer, only report what's explicitly stated" discipline
are carried over from the full pipeline's `microbio-annotator-amr` skill.

In [ ]:
import json

# Edit prompt wording directly here, same as TRIAGE_PROMPTS (02_triage.ipynb's section b). Re-run this cell after
# any edit. CURATION_PIPELINE is the fixed stage order -- curation is a straight-line chain, not
# a branching flowchart like TRIAGE_FLOWCHART, since every stage always runs (there's no
# yes/no gate here -- that's what triage already did). Only 3 extraction stages
# (strains/genes/phenotypes) plus a refinement pass -- no separate variant stage; AMR-specific
# fields that would otherwise sit on a variant (confers_resistance, antibiotic) sit on the gene
# instead, since this notebook's AMR lens cares about gene-level resistance signal, not
# individual sequence-level mutations.
#
# "antibiotic_aro_id" is NOT asked of the model -- like genus/species (below), it's a
# deterministic lookup against this repo's bundled aro.owl (CARD's Antibiotic Resistance
# Ontology), added after extraction by add_aro_id_column, matched on the model's own
# "antibiotic" text field against ARO term labels/synonyms.
CURATION_PIPELINE = ["strains", "genes", "phenotypes", "refinement"]

CURATION_PROMPTS = {
    "strains": {
        "target_section": ["IAO:0000305", "IAO:0000315", "IAO:0000317", "IAO:0000318", "IAO:0000314"],
        "output_format": "json_list",
        "prompt": (
            "List ALL bacterial organisms/strains mentioned in this paper. Scan Methods first "
            "(sections like 'Bacterial strains', 'Strains and plasmids', 'Culture conditions'), "
            "then Results and the rest of the paper. When in doubt about relevance, include it -- "
            "over-extraction is preferred to under-extraction at this stage.\n\n"
            "For each organism, return an object with:\n"
            '- "strain_name": most specific designation available (strain code, sub-strain, lab '
            "designation). Never truncate to species name alone if a strain identifier exists. "
            "Null if no strain-level identifier is given.\n"
            '- "taxon_name": the organism\'s scientific name as written in the paper.\n'
            '- "biome": ONLY if explicitly stated -- in vivo/ex vivo/in vitro context or a named '
            "cell line (e.g. 'HT29', 'Caco-2', 'murine infection model'). Do NOT infer from "
            "organism type alone. Null if not stated.\n"
            '- "isolation_source": the specific body site/material the organism was isolated from, '
            "if stated (e.g. 'nasopharynx', 'blood culture', 'soil', 'wound drainage fluid'). Null "
            "if not stated.\n"
            '- "host": the host organism it was isolated from, if stated (e.g. '
            "'Homo sapiens', 'Bos taurus'). Null if not stated or if this is an environmental/lab "
            "isolate with no host.\n"
            '- "host_age", "host_sex": only if explicitly stated for this isolate\'s host. Null '
            "otherwise.\n"
            '- "country", "region": the country and geographical region/subregion of isolation, '
            "only if explicitly stated. Null otherwise -- never infer from institution/author "
            "affiliation alone.\n"
            '- "latitude", "longitude": only if explicit coordinates are given in the paper. Null '
            "otherwise (do not look up or estimate coordinates from a place name).\n"
            '- "collection_year": the year (or date) the isolate was collected, if stated. Null '
            "otherwise.\n"
            '- "sra_accession", "biosample_id", "assembly_id": any SRA, BioSample, or '
            "Assembly accession number(s) given for this isolate, if stated. Null otherwise.\n"
            '- "evidence": a short verbatim quote (max ~30 words) supporting this entry.\n\n'
            "Do not infer or guess any field above -- null if not explicitly stated.\n\n"
            'Respond ONLY with a JSON array of these objects. Return [] if none found.'
        ),
    },
    "genes": {
        "target_section": ["IAO:0000317", "IAO:0000318", "IAO:0000319", "IAO:0000314"],
        "output_format": "json_list",
        "prompt": (
            "Using the strains list below as the only valid set of organisms, scan the full paper "
            "text for all genes, genomic regions, and genetic variants/mutations/knockouts "
            "mentioned for each organism (Methods: strain construction, plasmid tables, knockouts; "
            "Results: genotype-phenotype descriptions). Treat a described mutation/knockout of a "
            "gene as that same gene, not a separate entity -- note the modification in `evidence`.\n\n"
            "For each gene found, return an object with:\n"
            '- "organism_context": must exactly match a "taxon_name" or "strain_name" from the '
            "strains list below. Never introduce an organism not in that list.\n"
            '- "gene": locus tag if one appears anywhere in the paper for this gene, otherwise the '
            "gene symbol/name as written.\n"
            '- "gene_class": resistance gene family/class (e.g. beta-lactamase, efflux pump) ONLY '
            "if the gene name itself signals it or the paper states it. Null if not determinable.\n"
            '- "resistance_mechanism": mechanism as stated in the paper (target modification, '
            "enzymatic inactivation, efflux, reduced permeability, target bypass/protection). "
            "Null if not stated.\n"
            '- "confers_resistance": true ONLY if the paper explicitly links this gene (or a '
            "mutation/knockout of it) to a resistance/susceptibility change. Null if resistance is "
            "never discussed for this gene -- do not default to false.\n"
            '- "antibiotic": drug or drug class the paper ties to this gene\'s resistance effect, '
            "as written. Null if not stated.\n"
            '- "evidence": short verbatim quote.\n\n'
            "Do not infer or guess -- only report what the paper states, this applies doubly to "
            "the AMR-specific fields above.\n\n"
            "Strains list:\n{strains_json}\n\n"
            'Respond ONLY with a JSON array of these objects. Return [] if none found.'
        ),
    },
    "phenotypes": {
        "target_section": ["IAO:0000318", "IAO:0000319", "IAO:0000317", "IAO:0000314"],
        "output_format": "json_list",
        "prompt": (
            "Using the genes list below, scan the full paper text (Results: MIC tables, "
            "susceptibility panels, inhibition-zone data; Discussion; Methods: assay descriptions) "
            "for every experimentally reported phenotype linked to each gene. Do not skip minor or "
            "secondary readouts -- growth rates, sub-MIC effects, conditional observations all "
            "count. A gene with multiple measured phenotypes gets one object per phenotype. If a "
            "phenotype is reported for the organism as a whole with no specific gene implicated, "
            "leave \"gene\" null rather than guessing which gene list entry it belongs to.\n\n"
            "For each phenotype found, return an object with:\n"
            '- "gene": must exactly match a "gene" value from the genes list below, or null (see '
            "above). Never invent a new gene here.\n"
            '- "organism_context": must exactly match a "taxon_name" or "strain_name" from the '
            "strains/genes lists.\n"
            '- "phenotype_description": the specific observation, with numerical values if stated.\n'
            '- "phenotype_objective": what is being measured (e.g. MIC, inhibition zone diameter, '
            "growth rate).\n"
            '- "typing_method": the laboratory susceptibility-typing method used, if stated (e.g. '
            "'agar dilution', 'disk diffusion', 'broth microdilution', 'E-test'). Null if not "
            "stated.\n"
            '- "standard": the breakpoint/interpretive standard used, if stated (e.g. '
            "'CLSI', 'EUCAST', 'WHO', 'EU'). Null if not stated.\n"
            '- "platform": the named instrument/panel used for typing, if stated (e.g. '
            "'Sensititre', 'MicroScan', 'Vitek', 'Kirby-Bauer panel'). Null if not stated.\n"
            '- "antibiotic": drug or drug class tested, as written. Null if not stated.\n'
            '- "mic_value", "mic_unit": numeric MIC value and unit associated with this typing '
            "method, only if a number is actually reported. Both null if the paper gives only a "
            "qualitative R/S/I call.\n"
            '- "susceptibility_call": explicit R / S / I classification ONLY when the paper itself '
            "uses a categorical breakpoint call. Do not derive this yourself from a raw MIC number.\n"
            '- "comparison_to": control or reference strain, if stated.\n\n'
            "Do not infer or guess.\n\n"
            "Genes list:\n{genes_json}\n\n"
            'Respond ONLY with a JSON array of these objects. Return [] if none found.'
        ),
    },
    "refinement": {
        "target_section": ["IAO:0000305", "IAO:0000315", "IAO:0000317", "IAO:0000318", "IAO:0000319", "IAO:0000314"],
        "output_format": "json_dict",
        "prompt": (
            "Review this paper's full extraction below against the paper text one more time. "
            "Check: (1) any bacterial strain missing from the strains list? (2) any gene's "
            "organism_context that doesn't exactly match a strains entry? (3) any gene with "
            "confers_resistance=true that has no matching phenotype row? (4) any geographic field (country/region/lat/long) or accession "
            "number that looks inferred rather than explicitly stated? Report ONLY what you find "
            "wrong, do not re-extract everything.\n\n"
            "Full extraction:\nStrains:\n{strains_json}\n\nGenes:\n{genes_json}\n\n"
            "Phenotypes:\n{phenotypes_json}\n\n"
            'Respond ONLY with JSON in this shape: {"issues_found": ["one short sentence per '
            'issue, or empty list if none"]}'
        ),
    },
}

print("Curation pipeline:", CURATION_PIPELINE)


### Genus/species/organism -- parsed, not asked for

`genus`, `species`, and `organism` aren't LLM fields -- they're a deterministic split of
`taxon_name` (binomial nomenclature: first word is genus, second is species, the two together are
the organism), so there's no reason to spend a model call asking for something a `taxon_name`
already contains. `split_taxon_name` below is applied to `strains_df`/`all_strains_df` after
curation runs, not inside `CURATION_PROMPTS`.

In [ ]:
def split_taxon_name(taxon_name):
    """Deterministic genus/species split of a binomial taxon_name string, e.g.
    "Escherichia coli K-12" -> ("Escherichia", "coli", "Escherichia coli"). Returns
    (None, None, None) for a missing/empty taxon_name, and leaves `species` None for a
    single-word name (genus-only mention, or an abbreviated/non-binomial form)."""
    if not taxon_name or not isinstance(taxon_name, str):
        return None, None, None
    parts = taxon_name.strip().split()
    if not parts:
        return None, None, None
    genus = parts[0]
    species = parts[1] if len(parts) > 1 else None
    organism = f"{genus} {species}" if species else genus
    return genus, species, organism


def add_genus_species_columns(strains_df):
    """Adds genus/species/organism columns to a strains DataFrame in place, derived from its
    taxon_name column via split_taxon_name. No-op (returns df unchanged) if taxon_name is
    missing or the DataFrame is empty."""
    if strains_df.empty or "taxon_name" not in strains_df.columns:
        return strains_df
    parsed = strains_df["taxon_name"].apply(split_taxon_name)
    strains_df["genus"] = parsed.apply(lambda t: t[0])
    strains_df["species"] = parsed.apply(lambda t: t[1])
    strains_df["organism"] = parsed.apply(lambda t: t[2])
    return strains_df


### Antibiotic ARO id -- looked up, not asked for

Like `genus`/`species` above, `antibiotic_aro_id` isn't an LLM field -- the model was guessing
it from memory, and those guesses were wrong often enough to be worse than nothing. This repo
bundles `aro.owl` (CARD's Antibiotic Resistance Ontology) specifically so the id can be looked
up deterministically instead: `antibiotic_aro_id` is matched against the real ontology's term
labels/synonyms using the model's own `antibiotic` text field, after extraction, not asked for
in `CURATION_PROMPTS`.

In [ ]:
import re
import xml.etree.ElementTree as ET

ARO_OWL_PATH = "release_data/aro.owl"
_ARO_NS = {
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "oboInOwl": "http://www.geneontology.org/formats/oboInOwl#",
}


def _load_aro_name_to_id(path=ARO_OWL_PATH):
    """Parses aro.owl once into {lowercased label/synonym: "ARO:0000040"} -- each owl:Class
    carries one oboInOwl:id (the canonical "ARO:NNNNNNN" string), one rdfs:label, and zero or
    more oboInOwl:hasExactSynonym entries (trade names/salt forms, e.g. "streptomycin sulfate"
    alongside "streptomycin"). Returns {} if aro.owl isn't present -- callers degrade to
    leaving antibiotic_aro_id null rather than failing."""
    if not os.path.exists(path):
        print(f"  [ARO] {path} not found -- antibiotic_aro_id will be left null.")
        return {}
    root = ET.parse(path).getroot()
    name_to_id = {}
    for cls in root.findall("owl:Class", {"owl": "http://www.w3.org/2002/07/owl#"}):
        aro_id_el = cls.find("oboInOwl:id", _ARO_NS)
        if aro_id_el is None or not aro_id_el.text:
            continue
        aro_id = aro_id_el.text.strip()
        names = []
        label_el = cls.find("rdfs:label", _ARO_NS)
        if label_el is not None and label_el.text:
            names.append(label_el.text.strip())
        for syn_el in cls.findall("oboInOwl:hasExactSynonym", _ARO_NS):
            if syn_el.text:
                names.append(syn_el.text.strip())
        for name in names:
            name_to_id.setdefault(name.lower(), aro_id)
    print(f"  [ARO] Loaded {len(name_to_id)} name/synonym -> ARO id mappings from {path}.")
    return name_to_id


ARO_NAME_TO_ID = _load_aro_name_to_id()


def lookup_aro_id(antibiotic_name):
    """Exact (case-insensitive) match of a model-extracted "antibiotic" string against
    ARO_NAME_TO_ID. Returns None if there's no match or no name given -- no fuzzy matching,
    so a drug class phrase ("beta-lactams") or an unusual spelling won't resolve; that's a
    starting point to improve on, not a guarantee of full coverage."""
    if not antibiotic_name or not isinstance(antibiotic_name, str):
        return None
    return ARO_NAME_TO_ID.get(antibiotic_name.strip().lower())


def add_aro_id_column(df):
    """Adds/overwrites the antibiotic_aro_id column on a genes/phenotypes DataFrame via
    lookup_aro_id, in place of whatever the model may have guessed. No-op if the DataFrame is
    empty or has no antibiotic column."""
    if df.empty or "antibiotic" not in df.columns:
        return df
    df["antibiotic_aro_id"] = df["antibiotic"].apply(lookup_aro_id)
    return df

`call_llm_curation` reuses the same `_gemini_generate` plumbing defined above (section b) --
rate limiting, retries, daily cap -- but parses a JSON array or dict instead of a yes/no answer.
`run_curation` walks `CURATION_PIPELINE` in order, formatting each stage's prompt with the
accumulated output of earlier stages so the model can't invent an organism/gene that isn't
already on record.

### Choose which Gemini model runs curation

Curation can afford a different (e.g. stronger/slower) model than triage's `GEMINI_MODEL`
(section b) -- it's a handful of calls per paper instead of one per question, and a more capable
model matters more for a 15-field nested JSON schema than for a single yes/no answer. Pick one
from `GEMINI_MODEL_OPTIONS` below (or add your own); it's used by every curation call in this
section and is independent of `GEMINI_MODEL`.

In [ ]:
# Candidate Gemini models for curation -- check current availability/limits at
# https://ai.google.dev/gemini-api/docs/models before relying on any of these in a live session.
GEMINI_MODEL_OPTIONS = [
    "gemini-3.5-flash-lite",  # fastest/cheapest -- same default as section (b) above
    "gemini-3.5-flash",       # better instruction-following, still free-tier eligible
    "gemini-3-pro",           # strongest, lowest free-tier RPM/RPD -- use for small runs only
]

CURATION_GEMINI_MODEL = GEMINI_MODEL_OPTIONS[0]  # change the index (or paste a model id) to pick one

print(f"Curation will call: {CURATION_GEMINI_MODEL}")


In [ ]:
def _repair_truncated_array(array_text):
    """Best-effort recovery for a JSON array cut off mid-object -- hitting max_tokens with
    thinking mode on, or a long strains/genes list, truncates mid-string rather than at a
    clean boundary. Repeatedly closes the array at the last complete top-level '}' and
    retries, salvaging whatever complete objects were generated before the cutoff instead of
    discarding the whole stage over one unfinished entry at the tail. Returns None if nothing
    parseable is left."""
    idx = array_text.rfind("}")
    while idx != -1:
        candidate = array_text[:idx + 1] + "]"
        try:
            repaired = json.loads(candidate)
            if isinstance(repaired, list):
                return repaired
        except json.JSONDecodeError:
            pass
        idx = array_text.rfind("}", 0, idx)
    return None


def _extract_json_block(text):
    """Like _extract_json (section b) but handles both a JSON array ([...]) and a JSON
    object ({...}) response, since curation stages return lists except refinement. Falls back
    to _repair_truncated_array for an array that never got a closing ']' (truncated output)."""
    array_match = re.search(r"\[.*\]", text, re.DOTALL)
    if array_match:
        try:
            return json.loads(array_match.group(0))
        except json.JSONDecodeError:
            pass

    # No clean [...] match (likely truncated before the closing bracket) -- try to salvage
    # whatever complete objects exist between the first '[' and the cutoff.
    open_match = re.search(r"\[", text)
    if open_match:
        repaired = _repair_truncated_array(text[open_match.start():])
        if repaired:
            print(f"  [WARN] Model output was truncated -- recovered {len(repaired)} "
                  f"complete item(s) before the cutoff.")
            return repaired

    obj_match = re.search(r"\{.*\}", text, re.DOTALL)
    if obj_match:
        try:
            return json.loads(obj_match.group(0))
        except json.JSONDecodeError:
            pass

    print(f"  [WARN] Could not parse JSON from model output: {text[:200]!r}")
    return [] if "[" in text[:5] else {}


def _format_curation_prompt(prompt_def, paper_text, context):
    """Fill {strains_json}/{genes_json}/... placeholders in a CURATION_PROMPTS entry with the
    accumulated output of earlier stages (empty list/dict if that stage hasn't run yet). Uses
    plain str.replace, not str.format -- the prompt text itself contains literal JSON braces
    (e.g. {"strain_name": ...}) that .format() would misparse as format fields."""
    instructions = prompt_def["prompt"]
    for stage in CURATION_PIPELINE:
        placeholder = "{" + f"{stage}_json" + "}"
        instructions = instructions.replace(placeholder, json.dumps(context.get(stage, []))[:4000])
    return (
        f"{instructions}\n\n"
        f"Paper text:\n{paper_text[:30000]}\n\n"
        "Respond with ONLY the JSON described above -- no markdown fences, no extra prose."
    )


def _sanitize_curation_output(prompt_def, parsed):
    """Guarantees a stage's output actually matches its declared output_format, regardless of
    how malformed the model's response was -- a weaker/truncated model can return a JSON array
    of plain strings instead of objects (e.g. ["no data found"]), or a dict where a list was
    expected. Without this, a single bad row crashes every downstream `{"pmid": pmid, **row}`
    with "'str' object is not a mapping" instead of just being dropped with a warning."""
    expected = prompt_def.get("output_format")
    if expected == "json_list":
        if not isinstance(parsed, list):
            print(f"  [WARN] Expected a JSON list, got {type(parsed).__name__} -- using [].")
            return []
        bad = [r for r in parsed if not isinstance(r, dict)]
        if bad:
            print(f"  [WARN] Dropping {len(bad)} non-object item(s) from list output: {bad[:3]!r}")
        return [r for r in parsed if isinstance(r, dict)]
    if expected == "json_dict" and not isinstance(parsed, dict):
        # A model will sometimes wrap the one object it was asked for in a list -- unwrap
        # that case instead of discarding a perfectly good result.
        if isinstance(parsed, list) and len(parsed) == 1 and isinstance(parsed[0], dict):
            return parsed[0]
        # A weaker model (e.g. flash-lite) asked for {"issues_found": [...]} will sometimes
        # just return the bare list instead of wrapping it -- refinement is the only json_dict
        # stage and "issues_found" is its one key, so recover that shape instead of discarding
        # real findings.
        if isinstance(parsed, list):
            return {"issues_found": parsed}
        print(f"  [WARN] Expected a JSON object, got {type(parsed).__name__} -- using {{}}.")
        return {}
    return parsed


def call_llm_curation(prompt_def, paper_text, context, model=None, max_tokens=32768, max_retries=5):
    user_content = _format_curation_prompt(prompt_def, paper_text, context)
    content = _gemini_generate(user_content, model=model, max_tokens=max_tokens, max_retries=max_retries)
    parsed = _extract_json_block(content)
    return _sanitize_curation_output(prompt_def, parsed)


def run_curation(paper_text, model=None, verbose=True):
    """Runs every stage in CURATION_PIPELINE in order via Gemini, returning
    {stage_name: <parsed JSON>} -- a dict of lists for strains/genes/variants/phenotypes,
    and a dict for refinement. `model` defaults to CURATION_GEMINI_MODEL (set above) if not
    given -- pass a different Gemini model id to compare against it for the same paper."""
    if model is None:
        model = CURATION_GEMINI_MODEL
    context = {}
    for stage in CURATION_PIPELINE:
        prompt_def = CURATION_PROMPTS[stage]
        result = call_llm_curation(prompt_def, paper_text, context, model=model)
        context[stage] = result
        if verbose:
            n = len(result) if isinstance(result, list) else 1
            print(f"  [{stage}] {n} item(s)")
    return context


### Try it

Run curation on one sample paper (title+abstract, same input as `run_triage`) and tabulate each
stage as its own DataFrame. Full-text, section-aware curation works the same way as 02_triage.ipynb's
section (e): fetch `sections` via `fetch_sections_from_pmc`, then call `select_target_text(sections,
CURATION_PROMPTS[stage]["target_section"])` in place of `paper_text` before each stage -- left as
an exercise, since the wiring is identical to `run_triage_sectioned`.

In [ ]:
import pandas as pd 
import requests

demo_pmid = sample_pmids[0]
print(f"Curating PMID {demo_pmid} via {CURATION_GEMINI_MODEL}...")
curation_result = run_curation(papers[demo_pmid])

strains_df = add_genus_species_columns(pd.DataFrame(curation_result["strains"]))
genes_df = add_aro_id_column(pd.DataFrame(curation_result["genes"]))
phenotypes_df = add_aro_id_column(pd.DataFrame(curation_result["phenotypes"]))

print("\nRefinement check:", curation_result["refinement"])
strains_df


### Run curation across all sample papers

Loops `run_curation` over every PMID in `sample_pmids` (title+abstract, same input as the single-paper
cell above). Each stage's rows get tagged with their `pmid` so the per-paper results can be
concatenated into one combined DataFrame per entity type. This uses up to 5 Gemini calls per paper
(one per `CURATION_PIPELINE` stage) -- mind the free-tier rate/day caps from section (b) if
`sample_pmids` is long.

In [ ]:
all_curation_results = {}
all_strains_rows, all_genes_rows, all_phenotypes_rows = [], [], []

for pmid in sample_pmids:
    print(f"Curating PMID {pmid} via {CURATION_GEMINI_MODEL}...")
    try:
        result = run_curation(papers[pmid], verbose=False)
    except Exception as e:
        print(f"  [ERROR] Failed to curate PMID {pmid}: {e}")
        continue

    all_curation_results[pmid] = result
    for row in result["strains"]:
        all_strains_rows.append({"pmid": pmid, **row})
    for row in result["genes"]:
        all_genes_rows.append({"pmid": pmid, **row})
    for row in result["phenotypes"]:
        all_phenotypes_rows.append({"pmid": pmid, **row})

    issues = result["refinement"].get("issues_found", []) if isinstance(result["refinement"], dict) else []
    if issues:
        print(f"  [refinement] {len(issues)} issue(s) flagged for {pmid}: {issues}")

all_strains_df = add_genus_species_columns(pd.DataFrame(all_strains_rows))
all_genes_df = add_aro_id_column(pd.DataFrame(all_genes_rows))
all_phenotypes_df = add_aro_id_column(pd.DataFrame(all_phenotypes_rows))

print(f"\nCurated {len(all_curation_results)}/{len(sample_pmids)} papers.")
print(f"Strains: {len(all_strains_df)}, Genes: {len(all_genes_df)}, "
      f"Phenotypes: {len(all_phenotypes_df)}")
all_strains_df


In [ ]:
all_genes_df


In [ ]:
all_phenotypes_df

---
## f) Supplementary table mining

Curation (section f) only reads the main paper text. Many AMR papers put their actual
variant/antibiogram tables in a **Supplementary Table** or **Extended Data Table** instead --
never downloaded by anything above. This section mines exactly those files, using NCBI's own
Supplementary Material BioC API (`https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/supplmat.cgi/...`)
-- no other repository or package, and no local file-format parsing: NCBI has already extracted
text from `.xlsx`/`.docx`/`.pdf` supplementary files server-side, so `requests` alone is enough.

Whether a paper has anything to mine is decided by directly asking this API (`list_supp_files`,
below) whether it has any supplementary files on record for that paper -- **not** by
pattern-matching a pointer phrase like "see Supplementary Table 2" in the main text. A
regex-based gate would silently skip a paper whose wording doesn't match the pattern even
though it has real supplementary data; asking NCBI what it actually has on file is the ground
truth.

Gated behind `RUN_SUPP_MINING` below (default `False`), same as `RUN_REMOTE_COMPARISON` in 02_triage.ipynb -- so a top-to-bottom "Run All" never silently starts fetching supplementary files for
every paper in `sample_pmids`.

In [ ]:
RUN_SUPP_MINING = True  # set True to actually fetch/read supplementary files below

### Fetching supplementary files from NCBI's BioC API

`list_supp_files` lists a paper's supplementary files via NCBI's dedicated Supplementary
Material BioC API (`supplmat.cgi`) -- a separate endpoint from the full-text fetch (section d),
so it doesn't need to parse `<supplementary-material>` tags out of full-text XML/JSON at all.
Each entry already carries a direct per-file download URL, and NCBI has already run text
extraction server-side for xlsx/docx/pdf files, so `fetch_supp_file_text` gets back usable text
directly -- no local ZIP download, no `openpyxl`/`pdfplumber` parsing needed.

In [ ]:
import time

SUPP_REQUEST_DELAY = 0.34  # ~3 requests/sec, a conservative rate against NCBI's BioNLP API
_last_supp_request_time = 0.0
SUPP_USER_AGENT = {"User-Agent": "amr-workshop-supp-fetcher/1.0"}


def _supp_throttle():
    global _last_supp_request_time
    elapsed = time.time() - _last_supp_request_time
    if elapsed < SUPP_REQUEST_DELAY:
        time.sleep(SUPP_REQUEST_DELAY - elapsed)
    _last_supp_request_time = time.time()


def resolve_pmcid(doc_id):
    """Resolve doc_id (PMID or PMCID) to a PMCID string via pmid_to_pmcid (section d), or None."""
    doc_id = str(doc_id)
    if doc_id.upper().startswith("PMC"):
        return doc_id if doc_id.startswith("PMC") else doc_id.upper()
    return pmid_to_pmcid(doc_id)


def _supplmat_json(url, cache_name):
    """Shared fetch/cache/fallback logic for both supplmat.cgi calls below -- caches to
    bioc_cache/ (section d) so a re-run in this same working directory doesn't refetch, and
    falls back to the bundled sample_fulltext/ copy if NCBI is unreachable."""
    os.makedirs(BIOC_CACHE_DIR, exist_ok=True)
    cache_path = os.path.join(BIOC_CACHE_DIR, cache_name)
    if os.path.exists(cache_path):
        return json.load(open(cache_path))
    try:
        _supp_throttle()
        resp = requests.get(url, timeout=60, headers=SUPP_USER_AGENT)
        resp.raise_for_status()
        data = json.loads(resp.text)
        json.dump(data, open(cache_path, "w"))
        return data
    except Exception:
        local_path = f"{FALLBACK_DIR}/{cache_name}"
        if os.path.exists(local_path):
            return json.load(open(local_path))
        return None


def list_supp_files(doc_id):
    """List a paper's supplementary files via NCBI's PMC-OA Supplementary Material BioC API.
    Each entry carries "num" (use with fetch_supp_file_text below) and "original_file" (the
    file's real name/URL). Returns [] if the paper has no supplementary files on record, or
    isn't in PMC's Open Access subset."""
    pmcid = resolve_pmcid(doc_id)
    if not pmcid:
        print(f"[supp] No PMCID found for {doc_id} -- cannot list supp files")
        return []

    url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/supplmat.cgi/BioC_json/{pmcid}/list"
    data = _supplmat_json(url, f"{pmcid}_supp_list.json")
    entries = data if isinstance(data, list) else []
    print(f"[supp] {pmcid}: {len(entries)} supplementary file(s) on record")
    return entries


def fetch_supp_file_text(doc_id, num, max_chars=60000):
    """Get the extracted text of one supplementary file (by its "num" from list_supp_files)
    via supplmat.cgi's per-file BioC endpoint -- NCBI has already run xlsx/docx/pdf text
    extraction server-side, so this returns usable text directly."""
    pmcid = resolve_pmcid(doc_id)
    if not pmcid:
        return ""
    url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/supplmat.cgi/BioC_json/{pmcid}/{num}"
    data = _supplmat_json(url, f"{pmcid}_supp_{num}.json")
    if not data:
        return ""
    texts = []
    collections = data if isinstance(data, list) else [data]
    for collection in collections:
        for doc in collection.get("documents", []):
            for passage in doc.get("passages", []):
                text = passage.get("text", "")
                if text:
                    texts.append(text)
    return "\n".join(texts)[:max_chars]


print("Supplementary-file helpers loaded.")

### Mining every supplementary file NCBI has on record

`mine_supplementary_tables` lists every supplementary file via `list_supp_files` and fetches
each one's already-extracted text via `fetch_supp_file_text` -- no ZIP download or local file
parsing, since NCBI's BioC API returns the text directly. It does not try to parse a table into
structured rows itself -- that happens in the next cell, which passes the extracted text back
through `call_llm_curation`/`call_llm_curation_local` (section f) with the `phenotypes` prompt,
the same way the main text is curated, and appends the results to that paper's phenotype list.

In [ ]:
def mine_supplementary_tables(doc_id):
    """Fetch every supplementary file's extracted text for a paper via NCBI's dedicated
    Supplementary Material BioC API (list_supp_files/fetch_supp_file_text, above). Returns a
    list of {"label": ..., "filename": ..., "text": ...} -- empty list if the paper has no
    supplementary files on record."""
    entries = list_supp_files(doc_id)
    if not entries:
        print(f"[supp] {doc_id}: no supplementary files on record -- skipping.")
        return []

    results = []
    for e in entries:
        filename = e.get("original_file", "").rsplit("/", 1)[-1] or f"supp_{e.get('num')}"
        text = fetch_supp_file_text(doc_id, e["num"])
        results.append({"label": filename, "filename": filename, "text": text})

    n_read = sum(1 for r in results if r["text"])
    print(f"[supp] {doc_id}: {len(entries)} file(s) -- read text from {n_read}.")
    return results


print("mine_supplementary_tables ready.")

---
### Full per-paper pipeline: triage, then full-text curation, then supplementary curation

Runs each PMID in `sample_pmids` through, in order, before moving to the next paper:
1. **Triage** -- section-aware (`run_triage_sectioned`) if a PMC full-text record exists,
   else falls back to title+abstract (`run_triage`).
2. **Full-text curation** -- `run_curation_sectioned` below, which is `run_curation` (section e)
   reworked to pull each stage's input from the section(s) listed in its own `target_section`
   (via `select_target_text`, section d) instead of one flat title+abstract blob. Falls back to
   `run_curation` on title+abstract if there's no usable full text.
3. **Supplementary curation** -- if `RUN_SUPP_MINING` is `True` (above), `mine_supplementary_tables`
   is run once per paper and any matched table text is appended to every curation stage's input,
   so genes/phenotypes mentioned only in a supplementary table get a chance to be extracted.

Needs a PMC full-text record to get any benefit from steps 2-3 over section (f)'s abstract-only
run; papers without one still get triaged and curated on title+abstract.

In [ ]:
def run_curation_sectioned(sections, supp_text="", model=None, verbose=True):
    """Same stage walk as run_curation (section e), but each stage's main-text input is
    the section(s) named in its own target_section (select_target_text, section d) instead of
    one flat title+abstract blob.

    If supp_text is given, every stage ALSO runs a second, separate extraction call over
    supp_text alone (not concatenated onto the main-text input) and appends its results --
    string-concatenating a supplementary table dump onto the main text used to fight the
    same 6000-char budget in _format_curation_prompt, which either got silently truncated away
    or garbled the main text it was appended to. Two clean calls per stage, merged by list
    concatenation, avoids both. This does mean a gene/strain mentioned in both the main text
    and a supplementary table can appear twice -- acceptable for a workshop; de-duplication
    is a refinement-stage exercise, not handled here.

    Always calls Gemini (call_llm_curation, section e) -- this pipeline doesn't go through
    the local-model backend switch (section g); run that section's cells and call
    call_llm_curation_any yourself if you want to redo this with the local model."""
    model = model or CURATION_GEMINI_MODEL
    context = {}
    for stage in CURATION_PIPELINE:
        prompt_def = CURATION_PROMPTS[stage]
        stage_text = select_target_text(sections, prompt_def.get("target_section", []))
        result = call_llm_curation(prompt_def, stage_text, context, model=model)

        if supp_text:
            supp_result = call_llm_curation(prompt_def, supp_text, context, model=model)
            if isinstance(result, list) and isinstance(supp_result, list):
                result = result + supp_result
            elif isinstance(result, dict) and isinstance(supp_result, dict):
                result = {
                    "issues_found": list(result.get("issues_found", []))
                    + list(supp_result.get("issues_found", []))
                }

        context[stage] = result
        if verbose:
            n = len(result) if isinstance(result, list) else 1
            print(f"  [{stage}] {n} item(s)")
    return context


def run_curation_flat_with_supp(paper_text, supp_text="", model=None, verbose=True):
    """Same separate-calls-then-merge approach as run_curation_sectioned, for a paper with no
    PMC full-text record -- main text is title+abstract (section g) instead of per-stage
    sections, but supp_text still gets its own call per stage rather than being concatenated
    onto the main text. Always calls Gemini (call_llm_curation) -- same note as
    run_curation_sectioned above about the local-model backend switch (section g)."""
    model = model or CURATION_GEMINI_MODEL
    context = {}
    for stage in CURATION_PIPELINE:
        prompt_def = CURATION_PROMPTS[stage]
        result = call_llm_curation(prompt_def, paper_text, context, model=model)

        if supp_text:
            supp_result = call_llm_curation(prompt_def, supp_text, context, model=model)
            if isinstance(result, list) and isinstance(supp_result, list):
                result = result + supp_result
            elif isinstance(result, dict) and isinstance(supp_result, dict):
                result = {
                    "issues_found": list(result.get("issues_found", []))
                    + list(supp_result.get("issues_found", []))
                }

        context[stage] = result
        if verbose:
            n = len(result) if isinstance(result, list) else 1
            print(f"  [{stage}] {n} item(s)")
    return context


all_curation_results = {}
all_strains_rows, all_genes_rows, all_phenotypes_rows = [], [], []

for pmid in sample_pmids:
    pmcid = pmid_to_pmcid(pmid)
    sections = fetch_sections_from_pmc(pmcid) if pmcid else {}

    print(f"Triaging PMID {pmid}...")
    try:
        if sections:
            triage_result = run_triage_sectioned(sections, verbose=False)
        else:
            triage_result = run_triage(papers[pmid], verbose=False)
        # Note: triage always runs via Gemini here -- triage has its own backend choice
        # (02_triage.ipynb), separate from this notebook's curation backend (section g).
    except Exception as e:
        print(f"  [ERROR] Failed to triage PMID {pmid}: {e} -- skipping this paper.")
        all_curation_results[pmid] = {"triage": None, "skipped_reason": str(e)}
        continue
    print(f"  [triage] {triage_result}")

    # has_gene_genotype is recorded but doesn't gate the flowchart itself (both its
    # transitions go to check_amr_phenotype -- see TRIAGE_FLOWCHART above), so it shouldn't
    # gate curation either. A paper only needs to be about bacteria and have an AMR phenotype.
    triage_passed = all(
        triage_result.get(k) for k in ("is_bacteria", "has_amr_phenotype")
    )
    if not triage_passed:
        print(f"  [triage] PMID {pmid} did not pass triage -- skipping curation.")
        all_curation_results[pmid] = {"triage": triage_result}
        continue

    supp_text = ""
    if RUN_SUPP_MINING:
        supp_hits = mine_supplementary_tables(pmid)
        supp_text = "\n\n".join(
            f"[{s['label']} -- {s['filename']}]\n{s['text']}" for s in supp_hits if s["text"]
        )

    print(f"Curating PMID {pmid} via {CURATION_GEMINI_MODEL}...")
    try:
        if sections:
            result = run_curation_sectioned(sections, supp_text=supp_text, verbose=False)
        else:
            result = run_curation_flat_with_supp(papers[pmid], supp_text=supp_text, verbose=False)
    except Exception as e:
        print(f"  [ERROR] Failed to curate PMID {pmid}: {e}")
        continue

    all_curation_results[pmid] = {"triage": triage_result, **result}
    for row in result["strains"]:
        all_strains_rows.append({"pmid": pmid, **row})
    for row in result["genes"]:
        all_genes_rows.append({"pmid": pmid, **row})
    for row in result["phenotypes"]:
        all_phenotypes_rows.append({"pmid": pmid, **row})

    issues = result["refinement"].get("issues_found", []) if isinstance(result["refinement"], dict) else []
    if issues:
        print(f"  [refinement] {len(issues)} issue(s) flagged for {pmid}: {issues}")

all_strains_df = add_genus_species_columns(pd.DataFrame(all_strains_rows))
all_genes_df = add_aro_id_column(pd.DataFrame(all_genes_rows))
all_phenotypes_df = add_aro_id_column(pd.DataFrame(all_phenotypes_rows))

print(f"\nCurated {len(all_curation_results)}/{len(sample_pmids)} papers.")
print(f"Strains: {len(all_strains_df)}, Genes: {len(all_genes_df)}, "
      f"Phenotypes: {len(all_phenotypes_df)}")
all_strains_df

In [ ]:
all_genes_df

In [ ]:
all_phenotypes_df

---
## g) Advanced: host an open weight model

Everything above (sections e-f) ran entirely on Gemini. This section is optional -- it
loads a local open-weight model (same approach as `02_triage.ipynb`'s section f) so you can
compare it against Gemini on the same papers, or avoid an internet-dependent API call
entirely. The very next cell sets `CURATION_BACKEND` -- it defaults to `"gemini"`, so a
plain "Run All" never downloads/loads anything here. Set it to `"local"` *before* running
the model-loading cells below, then re-run whichever earlier cell you want to redo with
it, through `call_llm_curation_any`/`run_curation_any`.


In [ ]:
CURATION_BACKEND = "gemini"  # "gemini" or "local" -- set to "local" BEFORE running
                              # the cells below, otherwise the model download/load
                              # is skipped. Defaults to "gemini" so a plain "Run All"
                              # never pulls a multi-GB model onto your machine.


In [ ]:
if CURATION_BACKEND == "local":
    import platform

    IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"
    LOCAL_BACKEND = "mlx" if IS_APPLE_SILICON else "llama_cpp"

    # Set the model size ONCE here -- everything below (MLX repo, GGUF repo/filename) derives from
    # this, so there's nothing else to change if you want a smaller/faster model.
    QWEN_MODEL_SIZE = "14B"  # swap to "8B" or "4B" for a smaller/faster model

    print(f"Detected platform: {platform.system()} {platform.machine()} -> using the {LOCAL_BACKEND!r} backend")
    print(f"Model size: Qwen3-{QWEN_MODEL_SIZE}")
else:
    print("Skipping -- set CURATION_BACKEND = \"local\" above and rerun this cell to actually download/load the local model.")


In [ ]:
if CURATION_BACKEND == "local":
    if LOCAL_BACKEND == "mlx":
        !pip install -q mlx-lm
        MLX_MODEL_REPO = f"mlx-community/Qwen3-{QWEN_MODEL_SIZE}-4bit"
    else:
        !pip install -q llama-cpp-python huggingface_hub
        from huggingface_hub import list_repo_files

        LOCAL_MODEL_REPO = f"Qwen/Qwen3-{QWEN_MODEL_SIZE}-GGUF"
        print(f"GGUF files available in {LOCAL_MODEL_REPO}:")
        for f in list_repo_files(LOCAL_MODEL_REPO):
            if f.endswith(".gguf"):
                print(" ", f)
else:
    print("Skipping -- set CURATION_BACKEND = \"local\" above and rerun this cell to actually download/load the local model.")


In [ ]:
if CURATION_BACKEND == "local":
    if LOCAL_BACKEND == "llama_cpp":
        from huggingface_hub import hf_hub_download

        # Q4_K_M is the usual speed/quality sweet spot -- change QWEN_MODEL_SIZE above, not this,
        # if you want a different model size. Only swap this filename directly if you want a
        # different quantization of the same size (check the file list printed above).
        LOCAL_MODEL_FILE = f"Qwen3-{QWEN_MODEL_SIZE}-Q4_K_M.gguf"

        # hf_hub_download already checks the local HF cache and skips re-downloading if this file
        # was fetched before -- no separate try_to_load_from_cache check needed.
        model_path = hf_hub_download(repo_id=LOCAL_MODEL_REPO, filename=LOCAL_MODEL_FILE)
        print("Model path:", model_path)
    else:
        print("Using MLX -- no separate download step; the model downloads inside load() below.")
else:
    print("Skipping -- set CURATION_BACKEND = \"local\" above and rerun this cell to actually download/load the local model.")


In [ ]:
if CURATION_BACKEND == "local":
    if LOCAL_BACKEND == "mlx":
        from mlx_lm import load as mlx_load

        mlx_model, mlx_tokenizer = mlx_load(MLX_MODEL_REPO)
        print(f"Loaded {MLX_MODEL_REPO} via MLX.")
    else:
        import os
        from llama_cpp import Llama

        llm_local = Llama(
            model_path=model_path,
            # n_ctx=0 tells llama.cpp to use the model's OWN trained context length (from the
            # GGUF's metadata) instead of guessing a fixed number -- for Qwen3 that's 32K+. This
            # is the full budget prompt + thinking + answer all share; no more tuning this value
            # by hand when a long paper needs more room.
            n_ctx=0,
            n_threads=os.cpu_count(),
            verbose=False,
        )
        print("Loaded via llama.cpp.")
else:
    print("Skipping -- set CURATION_BACKEND = \"local\" above and rerun this cell to actually download/load the local model.")


In [ ]:
def _local_generate(user_content, max_tokens=200):
    """Backend-agnostic text generation -- hides the mlx-lm vs llama.cpp difference from
    call_llm_curation_local. Always suppresses the model's <think> block (Qwen3's reasoning
    models emit one by default) since curation only needs the final answer."""
    if LOCAL_BACKEND == "mlx":
        from mlx_lm import generate as mlx_generate

        messages = [{"role": "user", "content": user_content}]
        try:
            # enable_thinking is Qwen3's own chat-template switch for the <think> block.
            # Falls back to no kwarg if this tokenizer's template doesn't support it.
            prompt = mlx_tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, enable_thinking=False
            )
        except TypeError:
            prompt = mlx_tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        # mlx-lm's generate()/sampler API has changed across versions -- if this errors with an
        # unexpected keyword, check `pip show mlx-lm` and the mlx-lm README for the current
        # signature (e.g. newer versions take a `sampler=` object instead of `temp=`).
        return mlx_generate(mlx_model, mlx_tokenizer, prompt=prompt, max_tokens=max_tokens, verbose=False)
    else:
        # /no_think is Qwen3's own in-text switch to skip the <think> block.
        resp = llm_local.create_chat_completion(
            messages=[{"role": "user", "content": user_content + " /no_think"}],
            temperature=0,
            max_tokens=max_tokens,
        )
        return resp["choices"][0]["message"]["content"]


def strip_thinking(response):
    """Removes a <think>...</think> block (Qwen3 and similar reasoning models) if one slipped
    through despite /no_think -- e.g. an older model that doesn't honor it."""
    import re as _re_local

    match = _re_local.search(r"<think>.*?</think>", response, flags=_re_local.DOTALL)
    if match:
        return (response[:match.start()] + response[match.end():]).strip()
    return response.strip()


### Same chain, local model

`call_llm_curation_local`/`run_curation_local` call `_local_generate` + `strip_thinking` (section
e) instead of `_gemini_generate`. Needs section (e)'s local model already loaded. Each stage asks for one entity type at a time (not all four in
one call) specifically so a smaller local model's JSON output stays small enough to parse reliably
-- a 14B-or-smaller model asked for all four nested entity types in one response is much more
likely to truncate or emit invalid JSON.

In [ ]:
def call_llm_curation_local(prompt_def, paper_text, context, max_tokens=32768):
    # Generous flat budget -- n_ctx=0 above (llama.cpp) / the model's own default (MLX)
    # give plenty of context room, so this just needs to not be the bottleneck.
    user_content = _format_curation_prompt(prompt_def, paper_text, context)
    content = _local_generate(user_content, max_tokens=max_tokens)
    content = strip_thinking(content)
    parsed = _extract_json_block(content)
    return _sanitize_curation_output(prompt_def, parsed)


def run_curation_local(paper_text, verbose=True):
    """Same stage walk as run_curation, answered by the local model (section e) instead."""
    context = {}
    for stage in CURATION_PIPELINE:
        prompt_def = CURATION_PROMPTS[stage]
        result = call_llm_curation_local(prompt_def, paper_text, context)
        context[stage] = result
        if verbose:
            n = len(result) if isinstance(result, list) else 1
            print(f"  [{stage}] {n} item(s)")
    return context


### Choosing the backend: Gemini or local

`CURATION_BACKEND` (set at the top of this section, section g) picks which model
`call_llm_curation_any`/`run_curation_any` call -- `"gemini"` or `"local"` (needs this
section's model already loaded). Nothing above (the
single-paper demo, the multi-paper loop, or the full per-paper pipeline in section f) is
wired through these `_any` functions -- they always call Gemini directly. Use
`call_llm_curation_any`/`run_curation_any` yourself if you want to redo an earlier cell with
the local model instead.

In [ ]:
def call_llm_curation_any(prompt_def, paper_text, context, model=None, max_tokens=32768):
    """Dispatches to call_llm_curation (Gemini) or call_llm_curation_local depending on
    CURATION_BACKEND -- the one place every curation entry point goes through, so switching
    backends is one variable, not a find-and-replace across cells."""
    if CURATION_BACKEND == "local":
        return call_llm_curation_local(prompt_def, paper_text, context, max_tokens=max_tokens)
    return call_llm_curation(prompt_def, paper_text, context, model=model, max_tokens=max_tokens)


def run_curation_any(paper_text, model=None, verbose=True):
    """Same stage walk as run_curation/run_curation_local, dispatched per CURATION_BACKEND."""
    if model is None and CURATION_BACKEND == "gemini":
        model = CURATION_GEMINI_MODEL
    context = {}
    for stage in CURATION_PIPELINE:
        prompt_def = CURATION_PROMPTS[stage]
        result = call_llm_curation_any(prompt_def, paper_text, context, model=model)
        context[stage] = result
        if verbose:
            n = len(result) if isinstance(result, list) else 1
            print(f"  [{stage}] {n} item(s)")
    return context


print(f"Curation backend: {CURATION_BACKEND}")

---
## h) Clean up: free the local model's memory

Same as 02_triage.ipynb's section (g) -- only needed if you used `CURATION_BACKEND = "local"`
above.

In [ ]:
FREE_LOCAL_MODEL = True  # set True only once nothing later in this session still needs
                          # mlx_model/mlx_tokenizer/llm_local (local triage or curation) -- then
                          # re-run this cell to actually free them.

if not FREE_LOCAL_MODEL:
    print("FREE_LOCAL_MODEL is False -- skipping, local model (if loaded) stays resident. "
          "Set it to True above and re-run this cell when you're really done with it.")
else:
    import gc

    if "llm_local" in globals():
        del llm_local
        print("Freed llama.cpp model.")

    if "mlx_model" in globals():
        del mlx_model
        del mlx_tokenizer
        print("Freed MLX model.")

    gc.collect()

    if LOCAL_BACKEND == "mlx":
        try:
            import mlx.core as mx
            mx.clear_cache()
        except AttributeError:
            # Older mlx-core versions expose this under mx.metal instead.
            import mlx.core as mx
            mx.metal.clear_cache()

    print("Done.")
